<a href="https://colab.research.google.com/github/chibisova/study-tracker/blob/master/Copy_of_accesible_gaussian_splatting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Splatting - Accessible 3D Reconstruction

This notebook adapts the [original](https://github.com/graphdeco-inria/gaussian-splatting) reference implementation of "3D Gaussian Splatting for Real-Time Radiance Field Rendering" to create a simplified, educational and fully automated pipeline with a graphical interface for non-expert users.  
Developed by Daniel Cob Beirute, Instituto Tecnológico de Costa Rica (2025).

# Dependencies installation

In [ ]:
# Clone and install Guassian Splatting repository
%cd /content
!git clone --recursive https://github.com/DanielCob/accesible-gaussian-splatting.git
!pip install -q plyfile

%cd /content/accesible-gaussian-splatting
!pip install -q /content/accesible-gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/accesible-gaussian-splatting/submodules/simple-knn

/content
Cloning into 'accesible-gaussian-splatting'...
remote: Enumerating objects: 885, done.
remote: Total 885 (delta 0), reused 0 (delta 0), pack-reused 885 (from 1)
Receiving objects: 100% (885/885), 78.72 MiB | 23.75 MiB/s, done.
Resolving deltas: 100% (481/481), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/accesible-gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compr

In [ ]:
# Install dependencies
!sudo apt-get install -y colmap
!sudo apt-get install -y imagemagick
!sudo apt-get install -y xvfb # OpenGL workaround for Google Colab

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libamd2 libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libcamd2 libccolamd2 libceres2
  libcholmod3 libcolamd2 libcxsparse3 libdouble-conversion3 libevdev2
  libfreeimage3 libgflags2.2 libglew2.2 libgoogle-glog0v5 libgtk-3-0
  libgtk-3-bin libgtk-3-common libgudev-1.0-0 libilmbase25 libinput-bin
  libinput10 libjxr0 libmd4c0 libmetis5 libmtdev1 libopenexr25 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libraw20
  librsvg2-common libspqr2 libsuitesparseconfig5 libwacom-bin libwacom-common
  libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0
  libxcb-util1 libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxcomposite1
  libxkbcommon-x11-0 libxtst6 qt5-gtk-platformtheme qttranslations5-l10n
  session-migration
Suggested packages:
  gle

# Data upload point
(upload a short video orbiting a object or scene)

*Use remote dataset:

In [ ]:
# Skip video upload — use official sample images instead
!mkdir -p /content/accesible-gaussian-splatting/data/input

!wget https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip
!unzip -q tandt_db.zip

# Copy just the truck scene's raw images into the expected input folder
!cp tandt/truck/images/*.jpg /content/accesible-gaussian-splatting/data/input/

print("Sample images ready in /content/accesible-gaussian-splatting/data/input/")

--2026-06-21 07:16:08--  https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip
Resolving repo-sam.inria.fr (repo-sam.inria.fr)... 138.96.1.1
Connecting to repo-sam.inria.fr (repo-sam.inria.fr)|138.96.1.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 682628995 (651M) [application/zip]
Saving to: ‘tandt_db.zip’

tandt_db.zip        100%[===================>] 651.00M  13.5MB/s    in 50s     

2026-06-21 07:16:59 (13.0 MB/s) - ‘tandt_db.zip’ saved [682628995/682628995]

Sample images ready in /content/accesible-gaussian-splatting/data/input/


In [ ]:
##### SKIP THIS ######
from google.colab import files
import os

uploaded = files.upload()
# The user uploads a single video file
uploaded_filename = list(uploaded.keys())[0]

# Create the input directory if it doesn't exist
input_dir = '/content/accesible-gaussian-splatting/data/video'
os.makedirs(input_dir, exist_ok=True)

# Define the new path for the video
video_path = os.path.join(input_dir, uploaded_filename)

# Move the uploaded video to the input directory
os.rename(uploaded_filename, video_path)

print("Video uploaded and moved to:", video_path)

# Preproccesing

In [ ]:
# Create a directory to store the extracted frames
!mkdir /content/accesible-gaussian-splatting/data/input

# Use ffmpeg to extract frames from the uploaded video
!ffmpeg -i "{video_path}" -qscale:v 1 -qmin 1 -vf fps=2 /content/accesible-gaussian-splatting/data/input/%04d.jpg

print("Frames extracted to /content/gaussian-splatting/data/input/")

In [ ]:
!apt-get install -y imagemagick


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [ ]:
!ls -la /content/accesible-gaussian-splatting/data/
!ls -la /content/accesible-gaussian-splatting/data/sparse/

total 64
drwxr-xr-x 10 root root  4096 Jun 21 08:22 .
drwxr-xr-x 15 root root  4096 Jun 21 07:17 ..
drwxr-xr-x  3 root root  4096 Jun 21 08:14 distorted
drwxr-xr-x  2 root root 12288 Jun 21 08:22 images
drwxr-xr-x  2 root root  4096 Jun 21 08:22 images_2
drwxr-xr-x  2 root root  4096 Jun 21 08:22 images_4
drwxr-xr-x  2 root root  4096 Jun 21 08:22 images_8
drwxr-xr-x  2 root root 12288 Jun 21 07:17 input
-rw-r--r--  1 root root   648 Jun 21 08:22 run-colmap-geometric.sh
-rw-r--r--  1 root root   651 Jun 21 08:22 run-colmap-photometric.sh
drwxr-xr-x  3 root root  4096 Jun 21 08:22 sparse
drwxr-xr-x  5 root root  4096 Jun 21 08:22 stereo
total 12
drwxr-xr-x  3 root root 4096 Jun 21 08:22 .
drwxr-xr-x 10 root root 4096 Jun 21 08:22 ..
drwxr-xr-x  2 root root 4096 Jun 21 08:22 0


In [ ]:
!ls -la /content/accesible-gaussian-splatting/data/sparse/0/

total 32280
drwxr-xr-x 2 root root     4096 Jun 21 08:22 .
drwxr-xr-x 3 root root     4096 Jun 21 08:22 ..
-rw-r--r-- 1 root root       64 Jun 21 08:22 cameras.bin
-rw-r--r-- 1 root root 26206257 Jun 21 08:22 images.bin
-rw-r--r-- 1 root root  5310909 Jun 21 08:22 points3D.bin
-rw-r--r-- 1 root root  1516526 Jun 21 08:22 points3D.ply


In [ ]:
%cd /content/accesible-gaussian-splatting
!python convert.py -s /content/accesible-gaussian-splatting/data/ --resize

Streaming output truncated to the last 5000 lines.
         Time : 0.00716686 [s]
 Initial cost : 0.520301 [px]
   Final cost : 0.353994 [px]
  Termination : Convergence

  => Continued observations: 488
  => Added observations: 255

Bundle adjustment report
------------------------
    Residuals : 10464
   Parameters : 3791
   Iterations : 17
         Time : 0.249847 [s]
 Initial cost : 0.273875 [px]
   Final cost : 0.258058 [px]
  Termination : Convergence

  => Merged observations: 25
  => Completed observations: 8
  => Filtered observations: 10
  => Changed observations: 0.007367

Bundle adjustment report
------------------------
    Residuals : 10464
   Parameters : 3782
   Iterations : 3
         Time : 0.0457311 [s]
 Initial cost : 0.287757 [px]
   Final cost : 0.28207 [px]
  Termination : Convergence

  => Merged observations: 0
  => Completed observations: 0
  => Filtered observations: 0
  => Changed observations: 0.000000

Registering image #162 (162)

  => Image sees 382 / 1

In [ ]:
import os
print(os.path.exists("/content/accesible-gaussian-splatting/data/sparse"))
print(os.path.exists("/content/accesible-gaussian-splatting/data/images"))

True
True


# Training

In [ ]:
%cd /content/accesible-gaussian-splatting
!python train.py -s /content/accesible-gaussian-splatting/data/

/content/accesible-gaussian-splatting
2026-06-21 08:44:01.937451: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing 
Output folder: ./output/89692856-a [21/06 08:44:06]
Reading camera 251/251 [21/06 08:44:07]
Loading Training Cameras [21/06 08:44:07]
Loading Test Cameras [21/06 08:44:12]
Number of points at initialisation :  56159 [21/06 08:44:12]
Training progress: 100% 7000/7000 [08:53<00:00, 13.12it/s, Loss=0.0794303, Depth Loss=0.0000000]

[ITER 7000] Evaluating train: L1 0.04381655342876911 PSNR 22.329563903808594 [21/06 08:53:13]

[ITER 7000] Saving Gaussians [21/06 08:53:13]

Training complete. [21/06 08:53:26]


In [ ]:
!ls /content/accesible-gaussian-splatting/output/


89692856-a  e6eefa9b-5	e7c8a749-2


In [ ]:
!ls -la /content/accesible-gaussian-splatting/output/89692856-a/
!ls -la /content/accesible-gaussian-splatting/output/e6eefa9b-5/
!ls -la /content/accesible-gaussian-splatting/output/e7c8a749-2/

total 9292
drwxr-xr-x 3 root root    4096 Jun 21 08:53 .
drwxr-xr-x 5 root root    4096 Jun 21 08:44 ..
-rw-r--r-- 1 root root  100558 Jun 21 08:44 cameras.json
-rw-r--r-- 1 root root     235 Jun 21 08:44 cfg_args
-rw-r--r-- 1 root root 7823968 Jun 21 08:53 events.out.tfevents.1782031446.c4cdf4959f71.27073.0
-rw-r--r-- 1 root root   47692 Jun 21 08:53 exposure.json
-rw-r--r-- 1 root root 1516526 Jun 21 08:44 input.ply
drwxr-xr-x 3 root root    4096 Jun 21 08:53 point_cloud
total 15872
drwxr-xr-x 3 root root     4096 Jun 21 08:32 .
drwxr-xr-x 5 root root     4096 Jun 21 08:44 ..
-rw-r--r-- 1 root root   100558 Jun 21 08:22 cameras.json
-rw-r--r-- 1 root root      234 Jun 21 08:22 cfg_args
-rw-r--r-- 1 root root 14565443 Jun 21 08:32 events.out.tfevents.1782030152.c4cdf4959f71.21695.0
-rw-r--r-- 1 root root    41612 Jun 21 08:32 exposure.json
-rw-r--r-- 1 root root  1516526 Jun 21 08:22 input.ply
drwxr-xr-x 3 root root     4096 Jun 21 08:31 point_cloud
total 16
drwxr-xr-x 2 root root 409

In [ ]:
!ls /content/accesible-gaussian-splatting/output/89692856-a/point_cloud/

iteration_7000


In [ ]:
!python render.py -m /content/accesible-gaussian-splatting/output/89692856-a

Looking for config file in /content/accesible-gaussian-splatting/output/89692856-a/cfg_args
Config file found: /content/accesible-gaussian-splatting/output/89692856-a/cfg_args
Rendering /content/accesible-gaussian-splatting/output/89692856-a
Loading trained model at iteration 7000 [21/06 09:00:46]
Reading camera 251/251 [21/06 09:00:47]
Loading Training Cameras [21/06 09:00:47]
Loading Test Cameras [21/06 09:00:53]
Rendering progress: 100% 251/251 [02:07<00:00,  1.97it/s]
Rendering progress: 0it [00:00, ?it/s]


# Training with Evaluation (Train/Test split)

In [ ]:
%cd /content/accesible-gaussian-splatting
!python train.py -s /content/accesible-gaussian-splatting/data/ --eval
!python render.py -m /content/accesible-gaussian-splatting/data/
!python metrics.py -m /content/accesible-gaussian-splatting/data/

/content/accesible-gaussian-splatting
2026-06-21 08:22:27.496360: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing 
Output folder: ./output/e6eefa9b-5 [21/06 08:22:32]
------------LLFF HOLD------------- [21/06 08:22:33]
Reading camera 251/251 [21/06 08:22:33]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [21/06 08:22:33]
Loading Training Cameras [21/06 08:22:33]
Loading Test Cameras [21/06 08:22:38]
Number of points at initialisation :  56159 [21/06 08:22:39]
Training progress: 100% 7000/7000 [08:52<00:00, 13.13it/s, Loss=0.0814199, Depth Loss=0.0000000]

[ITER 7000] Evaluating test: L1 0.05030633159913123 PSNR 21.753430366516113 [21/06 08:31:44]

[ITER 7000] Evaluating train: L1 0.03954255357384682

# Exporting output
(.zip can later be visualized using other visualization tools like the one provided by [Camenduru](https://colab.research.google.com/github/camenduru/gaussian-splatting-colab/blob/main/gaussian_splatting_viewer_colab.ipynb) )

In [ ]:
import shutil
import os
from google.colab import files

output_dir = '/content/accesible-gaussian-splatting/output'
compressed_output_path = '/content/accesible-gaussian-splatting/output.zip'

# Compress the output folder
shutil.make_archive(compressed_output_path.replace('.zip', ''), 'zip', output_dir)

# Provide a download link
files.download(compressed_output_path)

print(f"Output folder compressed to {compressed_output_path} and ready for download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Output folder compressed to /content/accesible-gaussian-splatting/output.zip and ready for download.
